# GlobalMart — Bronze Layer Deep Analysis
**Purpose:** Systematically profile every Bronze table
**Tables analysed:**
- `globalmart_dev.bronze.customers`
- `globalmart_dev.bronze.orders`
- `globalmart_dev.bronze.transactions`
- `globalmart_dev.bronze.returns`
- `globalmart_dev.bronze.products`
- `globalmart_dev.bronze.vendors`

**Output** A documented findings report that drives every
transformation decision in the Silver notebook.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [0]:
# Fully qualified table references
CATALOG = "globalmart_dev"
SCHEMA  = "bronze"

def tbl(name):
    return f"{CATALOG}.{SCHEMA}.{name}"

def section(title):
    print(f"\n{'─'*72}\n  {title}\n{'─'*72}")

In [0]:
def profile_table(table_name):
    df    = spark.table(tbl(table_name))
    total = df.count()
    rescued_total = (
        df.where(F.col("_rescued_data").isNotNull()).count()
        if "_rescued_data" in df.columns else 0
    )

    print(f"\n{'='*72}")
    print(f"  TABLE : {table_name}")
    print(f"  ROWS  : {total:,}   |   ROWS WITH RESCUED DATA: {rescued_total:,}")
    print(f"{'='*72}")
    print(f"  {'COLUMN':<38} {'SPARK TYPE':<14} {'NULLS':>8} {'NULL%':>7} {'DISTINCT':>10}")
    print(f"  {'─'*38} {'─'*14} {'─'*8} {'─'*7} {'─'*10}")

    for field in df.schema.fields:
        col_name  = field.name
        col_type  = type(field.dataType).__name__.replace("Type","")
        null_cnt  = df.where(F.col(col_name).isNull()).count()
        null_pct  = round(null_cnt / total * 100, 1) if total > 0 else 0
        distinct  = df.select(col_name).distinct().count()

        flag = ""
        if null_pct == 100:
            flag = " ◄ 100% NULL"
        elif null_pct > 80:
            flag = " ◄ mostly null"
        elif distinct == total and null_cnt == 0:
            flag = " ◄ unique — candidate key"
        elif distinct == 1:
            flag = " ◄ single value — constant or bug"
        elif col_name == "_rescued_data" and null_cnt < total:
            flag = f" ◄ {total - null_cnt:,} rows rescued — type mismatch in source"

        print(f"  {col_name:<38} {col_type:<14} {null_cnt:>8,} {null_pct:>6.1f}% {distinct:>10,}{flag}")

    print()
    return df

In [0]:
df_customers    = profile_table("customers")
df_orders       = profile_table("orders")
df_transactions = profile_table("transactions")
df_returns      = profile_table("returns")
df_products     = profile_table("products")
df_vendors      = profile_table("vendors")

In [0]:
section("2.1  Source file breakdown — all tables")

for name, df in [
    ("customers",    df_customers),
    ("orders",       df_orders),
    ("transactions", df_transactions),
    ("returns",      df_returns),
    ("products",     df_products),
    ("vendors",      df_vendors),
]:
    print(f"\n── {name} ──")
    df.groupBy("_source_file_name", "source_region") \
      .agg(
          F.count("*").alias("rows"),
          F.min("_ingested_at").alias("first_ingested"),
          F.max("_ingested_at").alias("last_ingested"),
      ) \
      .orderBy("_source_file_name") \
      .show(truncate=False)

In [0]:
section("3.1  Rescued data — which tables and which files are affected")

for name, df in [
    ("customers",    df_customers),
    ("orders",       df_orders),
    ("transactions", df_transactions),
    ("returns",      df_returns),
    ("products",     df_products),
    ("vendors",      df_vendors),
]:
    if "_rescued_data" not in df.columns:
        continue
    rescued = df.where(F.col("_rescued_data").isNotNull())
    cnt = rescued.count()
    if cnt == 0:
        print(f"  {name}: no rescued rows")
        continue

    print(f"\n── {name}: {cnt:,} rescued rows ──")
    rescued.groupBy("_source_file_name") \
           .agg(F.count("*").alias("rescued_rows")) \
           .show(truncate=False)


In [0]:
section("3.2  Rescued data — what JSON keys are inside? (these are the lost columns)")
# Parsed rescued JSON and extract its keys.
# Every key here is a column value that failed type inference and needs to be recovered in Silver via get_json_object().

for name, df in [
    ("customers",    df_customers),
    ("transactions", df_transactions),
    ("returns",      df_returns),
]:
    rescued = df.where(F.col("_rescued_data").isNotNull())
    if rescued.count() == 0:
        continue

    print(f"\n── {name} — sample rescued JSON payloads ──")
    rescued.select("_source_file_name", "_rescued_data") \
           .show(5, truncate=False)

    # Extract all keys that appear inside the rescued JSON.
    sample_json = rescued.select("_rescued_data").limit(1).collect()[0][0]
    print(f"  Inferred schema of rescued JSON: {spark.range(1).select(F.schema_of_json(F.lit(sample_json))).collect()[0][0]}")
    print()


In [0]:
section("3.3  Rescued data — can we always recover the value?")
# For each rescued column, parse it back and count how many
# succeed vs fail. Fail count = unrecoverable rows → quarantine in Silver.

# transactions: Order_ID and Product_ID
txn_rescued = df_transactions.where(F.col("_rescued_data").isNotNull()) \
    .withColumn("parsed", F.from_json(F.col("_rescued_data"),
        "Order_ID STRING, Product_ID STRING, _file_path STRING"))

print("transactions — rescued Order_ID recovery rate:")
txn_rescued.agg(
    F.count("*").alias("total_rescued"),
    F.count(F.when(F.col("parsed.Order_ID").isNotNull(),   1)).alias("Order_ID_recovered"),
    F.count(F.when(F.col("parsed.Product_ID").isNotNull(), 1)).alias("Product_ID_recovered"),
    F.count(F.when(F.col("parsed.Order_ID").isNull(),      1)).alias("UNRECOVERABLE"),
).show(truncate=False)

# returns: refund_amount arrives as "$130.65" — strip "$" and cast
ret_rescued = df_returns.where(F.col("_rescued_data").isNotNull()) \
    .withColumn("parsed", F.from_json(F.col("_rescued_data"),
        "refund_amount STRING, _file_path STRING"))

print("returns — rescued refund_amount recovery rate (after stripping '$'):")
ret_rescued.withColumn(
    "refund_clean",
    F.regexp_replace(F.col("parsed.refund_amount"), r"[$,]", "").cast(T.DoubleType())
).agg(
    F.count("*").alias("total_rescued"),
    F.count(F.when(F.col("refund_clean").isNotNull(), 1)).alias("recovered_as_double"),
    F.count(F.when(F.col("refund_clean").isNull(),    1)).alias("UNRECOVERABLE"),
).show(truncate=False)


In [0]:

section("4.1  customers — which ID column does each source file use?")
# If a region uses customer_identifier but not customer_id, COALESCE must
# include customer_identifier. This cell tells us the exact mapping.

df_customers.groupBy("_source_file_name").agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("customer_id").isNotNull(),         1)).alias("customer_id"),
    F.count(F.when(F.col("CustomerID").isNotNull(),          1)).alias("CustomerID"),
    F.count(F.when(F.col("cust_id").isNotNull(),             1)).alias("cust_id"),
    F.count(F.when(F.col("customer_identifier").isNotNull(), 1)).alias("customer_identifier"),
).orderBy("_source_file_name").show(truncate=False)

# ─────────────────────────────────────────────────────────────────────────────
section("4.2  customers — which email / name / segment column does each file use?")

df_customers.groupBy("_source_file_name").agg(
    F.count("*").alias("total"),
    # email
    F.count(F.when(F.col("customer_email").isNotNull(), 1)).alias("customer_email"),
    F.count(F.when(F.col("email_address").isNotNull(),  1)).alias("email_address"),
    F.count(F.when(
        F.col("customer_email").isNull() & F.col("email_address").isNull(), 1
    )).alias("NO_email"),
    # name
    F.count(F.when(F.col("customer_name").isNotNull(), 1)).alias("customer_name"),
    F.count(F.when(F.col("full_name").isNotNull(),     1)).alias("full_name"),
    # segment
    F.count(F.when(F.col("segment").isNotNull(),          1)).alias("segment"),
    F.count(F.when(F.col("customer_segment").isNotNull(), 1)).alias("customer_segment"),
).orderBy("_source_file_name").show(truncate=False)


In [0]:
section("4.3  returns — which columns does each source file use?")
# returns_1.json and returns_2.json have completely different schemas.
# This cell maps exactly which file uses which column name so Silver
# can COALESCE the right pairs.

df_returns.groupBy("_source_file_name").agg(
    F.count("*").alias("total"),
    # order id
    F.count(F.when(F.col("OrderId").isNotNull(),        1)).alias("OrderId"),
    F.count(F.when(F.col("order_id").isNotNull(),       1)).alias("order_id"),
    # amount
    F.count(F.when(F.col("amount").isNotNull(),         1)).alias("amount"),
    F.count(F.when(F.col("refund_amount").isNotNull(),  1)).alias("refund_amount"),
    # date
    F.count(F.when(F.col("date_of_return").isNotNull(), 1)).alias("date_of_return"),
    F.count(F.when(F.col("return_date").isNotNull(),    1)).alias("return_date"),
    # reason
    F.count(F.when(F.col("reason").isNotNull(),         1)).alias("reason"),
    F.count(F.when(F.col("return_reason").isNotNull(),  1)).alias("return_reason"),
    # status
    F.count(F.when(F.col("status").isNotNull(),         1)).alias("status"),
    F.count(F.when(F.col("return_status").isNotNull(),  1)).alias("return_status"),
).orderBy("_source_file_name").show(truncate=False)


In [0]:
section("4.4  transactions — which Order / Product ID column does each file use?")
# transactions_3.csv has Order_ID (capital D) which clashed with Order_id
# (lowercase d) from other files — Auto Loader treated them as different columns
# and rescued one into _rescued_data. This cell confirms the mapping.

df_transactions.groupBy("_source_file_name").agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("Order_id").isNotNull(),   1)).alias("Order_id"),
    F.count(F.when(F.col("Order_ID").isNotNull(),   1)).alias("Order_ID_if_exists"),
    F.count(F.when(F.col("Product_id").isNotNull(), 1)).alias("Product_id"),
    F.count(F.when(F.col("_rescued_data").isNotNull(), 1)).alias("rescued_rows"),
).orderBy("_source_file_name").show(truncate=False)

In [0]:
section("5.1  customers — all distinct segment values across BOTH columns")
# We collect distinct values from both segment columns combined.
# Any value with count < 5% of the total is a candidate abbreviation or typo.

total_rows = df_customers.where(
    F.coalesce(F.col("segment"), F.col("customer_segment")).isNotNull()
).count()

df_customers.select(
    F.coalesce(F.col("segment"), F.col("customer_segment")).alias("raw_segment"),
    "_source_file_name"
).where(F.col("raw_segment").isNotNull()) \
 .groupBy("raw_segment") \
 .agg(
     F.count("*").alias("count"),
     (F.count("*") / total_rows * 100).alias("pct_of_total"),
     F.collect_set("_source_file_name").alias("source_files")
 ) \
 .orderBy(F.desc("count")) \
 .show(50, truncate=False)


In [0]:
section("5.2  customers — are city and state swapped in any file?")
# Strategy: build a reference set of known US state names.
# Then check if the 'city' column contains values that are actually state names.
# Any file where city contains state names has a column swap bug.

known_states = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
    "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
    "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
    "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New Hampshire","New Jersey","New Mexico","New York","North Carolina",
    "North Dakota","Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island",
    "South Carolina","South Dakota","Tennessee","Texas","Utah","Vermont",
    "Virginia","Washington","West Virginia","Wisconsin","Wyoming"
]

print("Files where 'city' column contains a US state name (swap detected):")
df_customers \
    .where(F.col("city").isin(known_states)) \
    .groupBy("_source_file_name") \
    .agg(
        F.count("*").alias("rows_with_state_in_city_col"),
        F.collect_set("city").alias("sample_state_values_in_city"),
    ) \
    .show(truncate=False)


In [0]:
section("5.3  orders — what date format patterns exist in each date column?")
# We don't assume the format. We regex-classify every value and count per file.
# Any format that is not one of the two main patterns is a new problem.

date_cols = [
    "order_purchase_date", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

def classify_date(col_name):
    return (
        F.when(F.col(col_name).isNull(), "NULL")
         .when(F.col(col_name).rlike(r"^\d{2}/\d{2}/\d{4}"),  "MM/DD/YYYY HH:mm")
         .when(F.col(col_name).rlike(r"^\d{4}-\d{2}-\d{2}"),  "YYYY-MM-DD")
         .when(F.col(col_name).rlike(r"^\d{2}-\d{2}-\d{4}"),  "DD-MM-YYYY")
         .otherwise("UNKNOWN — investigate")
    )

for col in date_cols:
    print(f"\n  Column: {col}")
    df_orders.withColumn("fmt", classify_date(col)) \
             .groupBy("_source_file_name", "fmt") \
             .agg(F.count("*").alias("count")) \
             .orderBy("_source_file_name", "fmt") \
             .show(truncate=False)


In [0]:
section("5.4  orders — are there logically impossible date sequences?")
# After Silver parses dates to timestamps, these sequences must hold:
#   purchase  ≤  approved  ≤  carrier_pickup  ≤  delivered_to_customer
# Violations = bad data that must be quarantined, not silently passed through.

orders_ts = df_orders \
    .withColumn("t_purchase",
        F.coalesce(
            F.try_to_timestamp("order_purchase_date", F.lit("MM/dd/yyyy HH:mm")),
            F.try_to_timestamp("order_purchase_date", F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp("order_purchase_date", F.lit("yyyy-MM-dd H:mm")),
        )
    ).withColumn("t_approved",
        F.coalesce(
            F.try_to_timestamp("order_approved_at", F.lit("MM/dd/yyyy HH:mm")),
            F.try_to_timestamp("order_approved_at", F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp("order_approved_at", F.lit("yyyy-MM-dd H:mm")),
        )
    ).withColumn("t_delivered",
        F.coalesce(
            F.try_to_timestamp("order_delivered_customer_date", F.lit("MM/dd/yyyy HH:mm")),
            F.try_to_timestamp("order_delivered_customer_date", F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp("order_delivered_customer_date", F.lit("yyyy-MM-dd H:mm")),
        )
    )

checks = {
    "approved BEFORE purchase"  : (F.col("t_approved")  < F.col("t_purchase")),
    "delivered BEFORE approved" : (F.col("t_delivered") < F.col("t_approved")),
    "purchase date in future"   : (F.col("t_purchase")  > F.current_timestamp()),
}

for label, condition in checks.items():
    cnt = orders_ts.where(
        F.col("t_purchase").isNotNull() &
        F.col("t_approved").isNotNull() &
        condition
    ).count()
    print(f"  {label}: {cnt:,} rows")

In [0]:
section("5.5  transactions — what are the actual data types of Sales, Quantity, discount?")
# Spark may have inferred STRING for numeric columns due to mixed formats.
# We test castability: if cast fails → non-numeric junk in source.
# We also find the value distribution to catch out-of-range values.

print("  Sales — castability and range:")
df_transactions.select(
    F.expr("try_cast(Sales as DOUBLE)").alias("Sales_num")
).agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("Sales_num").isNull(), 1)).alias("non_numeric_Sales"),
    F.min("Sales_num").alias("min_Sales"),
    F.max("Sales_num").alias("max_Sales"),
    F.count(F.when(F.col("Sales_num") <= 0, 1)).alias("zero_or_negative_Sales"),
).show(truncate=False)

print("  Quantity — castability and range:")
df_transactions.select(
    F.expr("try_cast(Quantity as INT)").alias("Quantity_num")
).agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("Quantity_num").isNull(), 1)).alias("non_numeric_Quantity"),
    F.min("Quantity_num").alias("min_Qty"),
    F.max("Quantity_num").alias("max_Qty"),
    F.count(F.when(F.col("Quantity_num") <= 0, 1)).alias("zero_or_negative_Qty"),
).show(truncate=False)

print("  discount — format variants per file (looking for '%' string format):")
df_transactions \
    .withColumn("discount_format",
        F.when(F.col("discount").isNull(),                          "NULL")
         .when(F.col("discount").rlike(r"%$"),                      "PERCENT_STRING  e.g. 40%")
         .when(F.expr("try_cast(discount as DOUBLE)").isNotNull(),   "DECIMAL_STRING  e.g. 0.4")
         .otherwise("OTHER — investigate")
    ) \
    .groupBy("_source_file_name", "discount_format") \
    .agg(F.count("*").alias("count")) \
    .orderBy("_source_file_name") \
    .show(truncate=False)

print("  discount — are any decimal values > 1.0 or < 0? (invalid range):")
df_transactions \
    .where(~F.col("discount").rlike(r"%$") & F.col("discount").isNotNull()) \
    .select(F.expr("try_cast(discount as DOUBLE)").alias("d")) \
    .agg(
        F.count(F.when(F.col("d") > 1.0, 1)).alias("discount_gt_1_INVALID"),
        F.count(F.when(F.col("d") < 0.0, 1)).alias("discount_negative_INVALID"),
        F.min("d").alias("min_discount"),
        F.max("d").alias("max_discount"),
    ).show(truncate=False)

In [0]:
section("5.6  transactions — profit: are negatives real or corrupt?")
# A negative profit can be legitimate (high discount, loss-leader sale).
# Or it can be a sign of corrupt data (wrong sign, data entry error).
# We cross-check: negative profit rows — do they have high discounts?
# If discount is high and profit is negative, it is a valid business loss.
# If discount is 0 and profit is negative, it is likely corrupt.

print("  Profit distribution per source file:")
df_transactions.groupBy("_source_file_name").agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("profit").isNull(),  1)).alias("null_profit"),
    F.count(F.when(F.col("profit") < 0,       1)).alias("negative_profit"),
    F.count(F.when(F.col("profit") == 0,      1)).alias("zero_profit"),
    F.count(F.when(F.col("profit") > 0,       1)).alias("positive_profit"),
    F.round(F.min("profit"), 2).alias("min"),
    F.round(F.max("profit"), 2).alias("max"),
).orderBy("_source_file_name").show(truncate=False)

print("  Negative profit rows — do they have discount > 0? (legitimacy check):")
df_transactions \
    .where(F.col("profit") < 0) \
    .withColumn("discount_d",
        F.regexp_replace(F.col("discount"), "%", "").cast(T.DoubleType())
    ) \
    .withColumn("discount_normalised",
        F.when(F.col("discount").rlike(r"%$"), F.col("discount_d") / 100)
         .otherwise(F.col("discount_d"))
    ) \
    .agg(
        F.count("*").alias("total_negative_profit_rows"),
        F.count(F.when(F.col("discount_normalised") > 0, 1)).alias("have_discount_EXPECTED"),
        F.count(F.when(F.col("discount_normalised") == 0, 1)).alias("no_discount_SUSPICIOUS"),
    ).show(truncate=False)

In [0]:
section("5.7  returns — date_of_return: all format variants including string 'NULL'")
# We do not know the formats in advance. Regex-classify every value.
# Any value that does not match a known date pattern is a problem for Silver.

df_returns \
    .withColumn("date_fmt",
        F.when(F.col("date_of_return").isNull(),                        "IS_SQL_NULL")
         .when(F.upper(F.trim(F.col("date_of_return"))) == "NULL",      "STRING_NULL  ← bug in source")
         .when(F.col("date_of_return").rlike(r"^\d{4}-\d{2}-\d{2}$"),  "YYYY-MM-DD")
         .when(F.col("date_of_return").rlike(r"^\d{2}-\d{2}-\d{4}$"),  "DD-MM-YYYY")
         .when(F.col("date_of_return").rlike(r"^\d{2}/\d{2}/\d{4}$"),  "MM/DD/YYYY")
         .otherwise("UNKNOWN — investigate")
    ) \
    .groupBy("date_fmt") \
    .agg(
        F.count("*").alias("count"),
        F.slice(F.collect_list("date_of_return"), 1, 3).alias("samples")
    ) \
    .show(truncate=False)

In [0]:
section("5.8  returns — NaN vs NULL in numeric amount columns")
# JSON NaN is NOT SQL NULL. In Spark, NaN propagates through aggregations
# (avg of [1, NaN, 3] = NaN, not 2). NULL is ignored by aggregations.
# Silver must explicitly convert NaN → NULL using nanvl() or isnan().
# This cell measures how many NaN values exist so we know the scope.

print("  refund_amount: NULL vs NaN breakdown:")
df_returns.agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("refund_amount").isNull(),                                       1)).alias("sql_null"),
    F.count(F.when(F.isnan(F.col("refund_amount")),                                       1)).alias("json_nan"),
    F.count(F.when(F.col("refund_amount").isNotNull() & ~F.isnan("refund_amount"),        1)).alias("real_value"),
).show(truncate=False)

print("  amount: NULL vs NaN breakdown:")
df_returns.agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("amount").isNull(),                                              1)).alias("sql_null"),
    F.count(F.when(F.isnan(F.col("amount")),                                              1)).alias("json_nan"),
    F.count(F.when(F.col("amount").isNotNull() & ~F.isnan("amount"),                     1)).alias("real_value"),
).show(truncate=False)


In [0]:
section("5.9  returns — all distinct reason and status values (garbage detection)")
# We collect every distinct value. Short or symbol values ("?", "N/A", "-")
# are garbage that Silver must convert to null. Casing variants must be normalised.

print("  'reason' distinct values:")
df_returns.where(F.col("reason").isNotNull()) \
    .groupBy("reason").agg(F.count("*").alias("count")) \
    .orderBy(F.desc("count")).show(50, truncate=False)

print("  'return_reason' distinct values:")
df_returns.where(F.col("return_reason").isNotNull()) \
    .groupBy("return_reason").agg(F.count("*").alias("count")) \
    .orderBy(F.desc("count")).show(50, truncate=False)

print("  'status' distinct values (per file — checking for casing/label variants):")
df_returns.groupBy("_source_file_name", "status") \
    .agg(F.count("*").alias("count")) \
    .orderBy("_source_file_name", F.desc("count")) \
    .show(truncate=False)

print("  'return_status' distinct values:")
df_returns.groupBy("_source_file_name", "return_status") \
    .agg(F.count("*").alias("count")) \
    .orderBy("_source_file_name", F.desc("count")) \
    .show(truncate=False)

In [0]:
section("5.10  orders — ship_mode and order_status: casing and label variants")

print("  ship_mode distinct values per file:")
df_orders.groupBy("_source_file_name", "ship_mode") \
    .agg(F.count("*").alias("count")) \
    .orderBy("_source_file_name", F.desc("count")) \
    .show(truncate=False)

print("  order_status distinct values per file:")
df_orders.groupBy("_source_file_name", "order_status") \
    .agg(F.count("*").alias("count")) \
    .orderBy("_source_file_name", F.desc("count")) \
    .show(truncate=False)

In [0]:
section("5.11  products — UPC format and product_id prefix distribution")
# UPC landed as string. What formats are present?
# product_id has a prefix (FUR, OFF, TEC). Do those prefixes match the
# category? Mismatches = catalog data quality issue.

print("  UPC format breakdown:")
df_products \
    .withColumn("upc_fmt",
        F.when(F.col("upc").isNull(),                         "NULL")
         .when(F.col("upc").rlike(r"^\d+\.\d+E[+\-]\d+$"),  "SCIENTIFIC_NOTATION")
         .when(F.col("upc").rlike(r"^\d{10,13}$"),           "PROPER_NUMERIC")
         .otherwise("OTHER")
    ) \
    .groupBy("upc_fmt").agg(F.count("*").alias("count")) \
    .show(truncate=False)

print("  product_id prefix vs first category — mismatches flagged:")
df_products \
    .withColumn("prefix",         F.split("product_id", "-").getItem(0)) \
    .withColumn("first_category", F.split("categories", ",").getItem(0)) \
    .groupBy("prefix", "first_category") \
    .agg(F.count("*").alias("count")) \
    .orderBy("prefix", F.desc("count")) \
    .show(100, truncate=False)

In [0]:
section("6.1  orders — order_id appearing in more than one region")

df_orders.groupBy("order_id") \
    .agg(
        F.countDistinct("source_region").alias("region_count"),
        F.count("*").alias("total_rows"),
        F.collect_set("source_region").alias("regions"),
    ) \
    .where(F.col("region_count") > 1) \
    .orderBy(F.desc("total_rows")) \
    .show(30, truncate=False)

In [0]:
section("6.2  transactions — same order_id transacted in multiple regions")
# For Region 3, Order_ID must be recovered from _rescued_data first.

txn_with_id = df_transactions.withColumn(
    "canonical_order_id",
    F.when(
        F.col("Order_id").isNull() & F.col("_rescued_data").isNotNull(),
        F.get_json_object("_rescued_data", "$.Order_ID")
    ).otherwise(F.coalesce("Order_id", "Order_ID"))
)

txn_with_id.groupBy("canonical_order_id") \
    .agg(
        F.countDistinct("source_region").alias("region_count"),
        F.count("*").alias("txn_rows"),
        F.round(F.sum(F.expr("try_cast(Sales as DOUBLE)")), 2).alias("sum_sales"),
        F.collect_set("source_region").alias("regions"),
    ) \
    .where(F.col("region_count") > 1) \
    .orderBy(F.desc("txn_rows")) \
    .show(30, truncate=False)

In [0]:
section("6.3  customers — same email appearing in multiple regions")

df_customers.select(
    F.lower(F.trim(
        F.coalesce("customer_email", "email_address")
    )).alias("email_norm"),
    F.coalesce("customer_id","CustomerID","cust_id","customer_identifier").alias("cid"),
    "source_region"
).where(F.col("email_norm").isNotNull()) \
 .groupBy("email_norm") \
 .agg(
     F.countDistinct("source_region").alias("region_count"),
     F.collect_set("cid").alias("ids"),
     F.collect_set("source_region").alias("regions"),
 ) \
 .where(F.col("region_count") > 1) \
 .orderBy(F.desc("region_count")) \
 .show(20, truncate=False)

In [0]:
section("6.4  returns — same order_id returned more than once (fraud signal)")

df_returns.withColumn(
    "coid", F.coalesce("order_id", "OrderId")
).groupBy("coid") \
 .agg(
     F.count("*").alias("return_count"),
     F.collect_set("_source_file_name").alias("files"),
 ) \
 .where(F.col("return_count") > 1) \
 .orderBy(F.desc("return_count")) \
 .show(20, truncate=False)

In [0]:
section("6.5  products — duplicate product_id check")

df_products.groupBy("product_id") \
    .agg(F.count("*").alias("count")) \
    .where(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

In [0]:
section("7.1  Build canonical ID sets for join checks")

canonical_customer_ids = df_customers.select(
    F.coalesce("customer_id","CustomerID","cust_id","customer_identifier")
     .alias("cid")
).distinct()

canonical_order_ids = df_orders.select("order_id").distinct()

vendor_ids = df_vendors.select("vendor_id").distinct()

In [0]:
section("7.2  orders — customer_id with no matching customer (orphaned orders)")

orphaned_orders = df_orders.join(
    canonical_customer_ids,
    df_orders.customer_id == canonical_customer_ids.cid,
    "left_anti"
)
print(f"  Orphaned orders (no customer match): {orphaned_orders.count():,}")
orphaned_orders.groupBy("source_region") \
    .agg(F.count("*").alias("count")) \
    .orderBy("source_region") \
    .show(truncate=False)

In [0]:
section("7.3  orders — vendor_id with no matching vendor")

orphaned_vendor = df_orders.join(
    vendor_ids,
    df_orders.vendor_id == vendor_ids.vendor_id,
    "left_anti"
)
print(f"  Orders with invalid vendor_id: {orphaned_vendor.count():,}")
orphaned_vendor.groupBy("source_region", "vendor_id") \
    .agg(F.count("*").alias("count")) \
    .show(truncate=False)

In [0]:
section("7.4  transactions — order_id with no matching order")

txn_orphaned = txn_with_id.join(
    canonical_order_ids,
    txn_with_id.canonical_order_id == canonical_order_ids.order_id,
    "left_anti"
)
print(f"  Orphaned transactions (no order match): {txn_orphaned.count():,}")
txn_orphaned.groupBy("source_region") \
    .agg(F.count("*").alias("count")) \
    .show(truncate=False)

In [0]:
section("7.5  returns — order_id with no matching order (highest fraud risk)")

ret_with_id = df_returns.withColumn(
    "coid", F.coalesce("order_id", "OrderId")
)
ret_orphaned = ret_with_id.join(
    canonical_order_ids,
    ret_with_id.coid == canonical_order_ids.order_id,
    "left_anti"
)
print(f"  Orphaned returns (no order match): {ret_orphaned.count():,}")
ret_orphaned.groupBy("_source_file_name") \
    .agg(F.count("*").alias("count")) \
    .show(truncate=False)

In [0]:
section("8 — Silver Readiness Checklist")

def check(label, fail_count, total, consequence="quarantine"):
    status = "✓ PASS" if fail_count == 0 else "✗ NEEDS FIX"
    pct    = round(fail_count / total * 100, 2) if total > 0 else 0
    print(f"  {status}  [{label}]  {fail_count:,} / {total:,} rows ({pct}%)  → {consequence}")

total_orders = df_orders.count()
total_txn    = df_transactions.count()
total_cust   = df_customers.count()
total_ret    = df_returns.count()
total_prod   = df_products.count()

In [0]:
print("\n  ── CUSTOMERS ──")
check("canonical_id resolvable",
    df_customers.where(
        F.col("customer_id").isNull() & F.col("CustomerID").isNull() &
        F.col("cust_id").isNull() & F.col("customer_identifier").isNull()
    ).count(), total_cust, "quarantine — no usable ID")

check("email present (warn only if R5)",
    df_customers.where(
        F.col("customer_email").isNull() & F.col("email_address").isNull()
    ).count(), total_cust, "warn + null accepted for R5")

check("city not a state name (R4 swap check)",
    df_customers.where(F.col("city").isin(known_states)).count(),
    total_cust, "swap city/state in Silver for R4")

In [0]:
print("\n  ── ORDERS ──")
check("order_id not null",
    df_orders.where(F.col("order_id").isNull()).count(),
    total_orders, "quarantine")

check("customer_id not null",
    df_orders.where(F.col("customer_id").isNull()).count(),
    total_orders, "quarantine")

check("purchase_date parseable",
    orders_ts.where(F.col("t_purchase").isNull()).count(),
    total_orders, "quarantine — cannot be used without date")

check("approved not before purchase",
    orders_ts.where(
        F.col("t_purchase").isNotNull() & F.col("t_approved").isNotNull() &
        (F.col("t_approved") < F.col("t_purchase"))
    ).count(), total_orders, "quarantine — impossible sequence")

In [0]:
print("\n  ── TRANSACTIONS ──")
check("order_id resolvable (incl. rescued)",
    txn_with_id.where(F.col("canonical_order_id").isNull()).count(),
    total_txn, "quarantine — cannot link to order")

check("Sales castable to DOUBLE",
    df_transactions.select(
        F.expr("try_cast(Sales as DOUBLE)").alias("Sales_num")
    ).where(F.col("Sales_num").isNull()).count(), 
    total_txn, "quarantine — revenue figure unreadable")

check("discount in valid range after normalisation",
    df_transactions.withColumn("d_norm",
        F.when(F.col("discount").rlike(r"%$"),
            F.expr("try_cast(regexp_replace(discount, '%', '') as DOUBLE)") / 100
        ).otherwise(F.expr("try_cast(discount as DOUBLE)"))
    ).where(
        F.col("d_norm").isNotNull() & ((F.col("d_norm") > 1.0) | (F.col("d_norm") < 0.0))
    ).count(), total_txn, "quarantine — discount outside 0–1 range")

check("negative profit with discount > 0 (valid losses)",
    df_transactions.where(F.col("profit") < 0).count(),
    total_txn, "WARN only — log in quality_issues table, keep in Silver")

In [0]:
print("\n  ── RETURNS ──")
check("order_id resolvable",
    df_returns.where(
        F.col("order_id").isNull() & F.col("OrderId").isNull()
    ).count(), total_ret, "quarantine")

check("date_of_return not a literal 'NULL' string",
    df_returns.where(
        F.upper(F.trim(F.col("date_of_return"))) == "NULL"
    ).count(), total_ret, "convert to SQL null in Silver")

check("reason not a garbage value",
    df_returns.where(
        F.col("reason").isin("?", "N/A", "-", "na", "none", "None")
    ).count(), total_ret, "convert to null in Silver")

check("refund_amount NaN converted",
    df_returns.where(F.isnan("refund_amount")).count(),
    total_ret, "nanvl(refund_amount, null) in Silver")

In [0]:
print("\n  ── PRODUCTS ──")
check("product_id not null",
    df_products.where(F.col("product_id").isNull()).count(),
    total_prod, "quarantine")

check("product_id unique",
    df_products.count() - df_products.select("product_id").distinct().count(),
    total_prod, "deduplicate in Silver")

print("\n  ── VENDORS ──")
check("vendor_id not null",
    df_vendors.where(F.col("vendor_id").isNull()).count(),
    df_vendors.count(), "quarantine")

print("\n  ── REFERENTIAL INTEGRITY ──")
check("orders → customers",      orphaned_orders.count(),  total_orders, "quarantine orphaned orders")
check("orders → vendors",        orphaned_vendor.count(),  total_orders, "warn + null vendor_id in Silver")
check("transactions → orders",   txn_orphaned.count(),     total_txn,    "quarantine — no parent order")
check("returns → orders",        ret_orphaned.count(),     total_ret,    "quarantine — highest fraud risk")
